In [19]:
import yfinance as yf
import pandas as pd

tickers = [
    "BTC-USD", "ETH-USD", "SOL-USD", "LINK-USD",
    "GC=F", "SI=F", "BZ=F", "NG=F", "HG=F", "ZC=F", "KC=F", "PA=F",
    "TLT", "IEF", "SHY", "TIP", "BNDX", "EMB", "VTC", "JNK", "IBGL.L", "BTP.MI", "MUB",
    "AAPL", "MSFT", "AMZN", "JNJ", "JPM", "XOM", "PG", "TSLA", "UNH", "BRK-B",
    "SAN.MC", "ITX.MC", "IBE.MC", "MC.PA", "SAP.DE", "ASML.AS", "SIE.DE", "NESN.SW",
    "AZN.L", "HSBA.L",
    "2330.TW", "7203.T", "BABA", "TCEHY", "RELIANCE.NS", "VALE", "BHP"
]

rows = []

for ticker in tickers:
    info = yf.Ticker(ticker).info

    rows.append({
        "ticker": ticker,
        "asset_name": info.get("shortName"),
        "sector": info.get("sector"),
        "exchange": info.get("exchange"),
        "currency": info.get("currency"),
        "beta": info.get("beta"),
        "dividend_yield": info.get("dividendYield")
    })

assets_df = pd.DataFrame(rows)


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BTP.MI"}}}


In [20]:
def asset_type(ticker):
    
    crypto = ["BTC-USD", "ETH-USD", "SOL-USD", "LINK-USD"]
    
    commodities = ["GC=F", "SI=F", "BZ=F", "NG=F", 
                   "HG=F", "ZC=F", "KC=F", "PA=F"]
    
    bonds = ["TLT", "IEF", "SHY", "TIP", "BNDX", "EMB", 
             "VTC", "JNK", "IBGL.L", "BTP.MI", "MUB"]
    
    if ticker in crypto:
        return "Crypto"
    
    elif ticker in commodities:
        return "Commodity"
    
    elif ticker in bonds:
        return "Bond"
    
    else:
        return "Equity"

assets_df["asset_type"] = assets_df["ticker"].apply(asset_type)

In [28]:
df_historico = pd.read_csv("historical_assets.csv")

grupby_activo = df_historico.groupby("ticker")

returns = grupby_activo["Close"].pct_change()

volatility = returns.groupby(df_historico["ticker"]).std()

volatility_anual = volatility * (252 ** 0.5)

assets_df["volatility"] = assets_df["ticker"].map(volatility_anual)

In [22]:
def max_drawdown(prices):
    cummax = prices.cummax()
    drawdown = (prices - cummax) / cummax
    return drawdown.min()

prices_df = df_historico.pivot(index="date", columns="ticker", values="Close")

mdd = prices_df.apply(max_drawdown)

assets_df["max_drawdown"] = assets_df["ticker"].map(mdd)

In [23]:
def avg_return(prices, years):
    return (prices.iloc[-1] / prices.iloc[-252*years] - 1)

assets_df["avg_return_1y"] = prices_df.apply(avg_return, years=1)
assets_df["avg_return_3y"] = prices_df.apply(avg_return, years=3)
assets_df["avg_return_5y"] = prices_df.apply(avg_return, years=5)

In [24]:
def risk_category(vol):
    if vol < 0.10:
        return "Low"
    elif vol < 0.25:
        return "Medium"
    return "High"

assets_df["risk_category"] = assets_df["volatility"].apply(risk_category)

In [25]:
def investment_style(row):
    if row["asset_type"] == "Bond":
        return "Defensive"
    if row["dividend_yield"] and row["dividend_yield"] > 0.03:
        return "Income"
    return "Growth"

assets_df["investment_style"] = assets_df.apply(investment_style, axis=1)

In [26]:
assets_df.to_csv("assets_info.csv", index=False)

In [27]:
assets_df.head()

,ticker,asset_name,sector,exchange,currency,beta,dividend_yield,asset_type,volatility,max_drawdown,avg_return_1y,avg_return_3y,avg_return_5y,risk_category,investment_style
0,BTC-USD,Bitcoin USD,NaN,CCC,USD,NaN,NaN,Crypto,0.035152,-0.833990,NaN,NaN,NaN,Low,Growth
1,ETH-USD,Ethereum USD,NaN,CCC,USD,NaN,NaN,Crypto,0.044972,-0.939625,NaN,NaN,NaN,Low,Growth
2,SOL-USD,Solana USD,NaN,CCC,USD,NaN,NaN,Crypto,0.063898,-0.962725,NaN,NaN,NaN,Low,Growth
3,LINK-USD,Chainlink USD,NaN,CCC,USD,NaN,NaN,Crypto,0.062802,-0.901929,NaN,NaN,NaN,Low,Growth
4,GC=F,Gold Apr 26,NaN,CMX,USD,NaN,NaN,Commodity,0.011129,-0.443638,NaN,NaN,NaN,Low,Growth
